In [ ]:
# bootstrap: Colab clone + local import of `rlcore` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "00-foundations/rl-and-thinking-models/rl-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "rlcore").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 03 · GRPO with verifiable rewards

**Tier:** T0 — CPU only, numpy, no network, under a minute. Every formula follows TRL's `GRPOTrainer`
(v1.14.0, verify) line for line, so each setting here maps onto a real `GRPOConfig`. A real GRPO step on a
tiny transformer is `thinking-lab` notebook `01_grpo_on_a_tiny_transformer`; one step with vLLM generating the
rollouts is its notebook `05_rl_rollouts_with_an_engine`.

## The one-minute version
**GRPO** drops PPO's value model: for each prompt, sample a **group** of G completions, score each with a
**verifier**, and give every token of completion i the advantage A_i = (r_i − mean r)/(std r + 1e-4). The
group is the baseline. A group that is all right or all wrong has zero advantage and teaches nothing — as
training succeeds, more groups go silent. The per-token loss is PPO's clipped surrogate
−min(ρA, clip(ρ, 1 − ε, 1 + ε)A) with ρ = π/π_old, plus β·k3, an always-positive KL estimator. With one
gradient step per batch (TRL's default μ = 1) the ratio is 1 and the clip never binds. Then the details that
decide what the model learns: averaging the loss **per sequence** under-weights long completions and bends the
update toward longer, truncated answers — the **length bias** DAPO and Dr. GRPO remove with a token-level or
constant normaliser; std scaling up-weights nearly-solved prompts; clipping caps how fast rare tokens can rise.
And the cost: G completions per prompt, generated by an inference engine, dominate the step. Primer:
`../PRIMER.md` §4 (and §1, §8 for the compute).

> **Exercise cells** contain `# YOUR CODE HERE` — replace it, then run the **Check** cell below it. A check prints ✅ when it passes. The finished version is in `solutions/`.

In [ ]:
import math

import numpy as np

from rlcore import Policy, SeqTask, ThinkTask, grpo, pg, workload

task = SeqTask("brackets", 8)
seqs = task.all_sequences()
ok = np.array([task.verify(s) for s in seqs])
ref = Policy.for_task(task)
for _ in range(2):                                   # the weak SFT reference of notebooks 01–02 (29.7% right)
    pg.sft_step(ref, [task.as_trajectory(s) for s in seqs if task.verify(s)], lr=1.0)


def diversity(pol):
    """P(correct) and the effective number of distinct correct answers, exp(entropy) over them."""
    p = pol.sequence_probs(task, seqs)
    pv = p[ok > 0] / p[ok > 0].sum()
    return float(p @ ok), float(np.exp(-(pv * np.log(pv)).sum()))

## Worked example 1 — the group is the baseline
TRL's code: `advantages = rewards - mean_grouped_rewards`, then `/ (std_rewards + 1e-4)`, the std with
Bessel's correction.

In [ ]:
for r in ([1, 0, 0, 1], [1, 0, 0, 0], [1, 1, 1, 1], [1] * 7 + [0], [1] * 4 + [0] * 4):
    print(f"rewards {r!s:26} → advantages {grpo.group_advantages([r])[0].round(4).tolist()}")

Read the last two rows. The single miss on a prompt the model nearly always solves gets −2.47; a miss on a
50/50 prompt gets −0.94. Dividing by the group's std up-weights prompts that are nearly solved or nearly
hopeless — Dr. GRPO's "difficulty bias"; `scale_rewards="none"` removes it. The all-correct group gets zero:
no signal, however many tokens it cost to generate.

## Worked example 2 — the clipped ratio, and when it binds
ρ = π(token)/π_old(token). For a positive advantage the objective stops growing once ρ > 1 + ε; for a negative
one, once ρ < 1 − ε. Where the clip binds the token's gradient is zero.

In [ ]:
ratio = np.array([0.7, 0.9, 1.0, 1.1, 1.3])
for adv in (1.0, -1.0):
    obj, d = grpo.clipped_surrogate(ratio, np.full(5, adv), 0.2)
    print(f"A = {adv:+.0f}: ratio {ratio.tolist()} → objective {obj.round(2).tolist()}, gradient {d.tolist()}")
for p_old in (0.01, 0.5, 0.9):
    print(f"π_old = {p_old:4}: one update can raise it to at most {min(1, p_old * 1.2):.4f} with ε = 0.2, "
          f"{min(1, p_old * 1.28):.4f} with ε_high = 0.28")

The cap is multiplicative, so a token the model rarely picks can barely rise in one update while a likely one
is capped only by 1 — DAPO's argument that symmetric clipping drives entropy down, and its fix **clip-higher**
(ε_low 0.2, ε_high 0.28). Note the TRL default `num_iterations=1`: sampling and updating with the same weights
makes ρ ≡ 1 and the clip inert; it matters only with μ > 1 or stale (off-policy) rollouts.

## Worked example 3 — the KL estimator k3
GRPO puts the KL in the loss, estimated per token from samples of π: k3 = π_ref/π − log(π_ref/π) − 1. Compare
it with the naive k1 = log(π/π_ref) on two small distributions whose true KL we know.

In [ ]:
p_, q_ = np.array([0.5, 0.3, 0.2]), np.array([0.3, 0.3, 0.4])
x = np.random.default_rng(0).choice(3, 100000, p=p_)
true_kl = float(np.sum(p_ * np.log(p_ / q_)))
for name, fn in (("k1", grpo.k1), ("k3", grpo.k3)):
    est = fn(np.log(p_[x]), np.log(q_[x]))
    print(f"{name}: mean {est.mean():.4f} (true {true_kl:.4f})  std {est.std():.3f}  min {est.min():+.3f}")
print("k3 at log(π_ref/π) = 0.1, −0.1, 0.5:", [round(float(grpo.k3(0.0, d)), 7) for d in (0.1, -0.1, 0.5)])

Both are unbiased; k3 is never negative and has a fraction of the spread, which matters when it is summed
over thousands of tokens. TRL's default is β = 0 — no reference model loaded at all (DAPO also drops the KL);
the DeepSeek-R1 recipe used β = 0.001 (per TRL's docs, verify).

## Worked example 4 — GRPO on the bracket task
Eight samples per prompt, two groups per step, μ = 4 gradient steps per batch so the clip is live.

In [ ]:
pol = ref.copy()
cfg = grpo.GRPOConfig(num_generations=8, num_iterations=4, lr=20.0)
hist = grpo.train_grpo(pol, task, np.random.default_rng(0), cfg, steps=100, prompts=(0, 0), log_every=20)
for h in hist:
    print(f"step {h['step']:3d}  correct {h['correct']:.2f}  zero-std groups {h['frac_reward_zero_std']:.2f}  "
          f"clipped tokens {h['clipped']:.2f}")
print("reference: P(correct) %.3f, %.1f effective correct answers" % diversity(ref))
print("after GRPO: P(correct) %.3f, %.1f effective correct answers" % diversity(pol))
for ec in (0.0, 0.05):                               # the other lever: pay for entropy in the reward
    p_ec = ref.copy()
    pg.train_reinforce(p_ec, task, np.random.default_rng(0), steps=300, batch=16, lr=0.5, entropy_coef=ec)
    print(f"REINFORCE, entropy bonus {ec}: P(correct) %.3f, %.1f effective correct answers" % diversity(p_ec))

Success goes to ~100% within a few dozen steps; after that almost every group is all-correct, advantages are
zero and learning stops — and the policy has concentrated on a few of the 14 correct strings. RL sharpens:
pass@1 up, diversity down. An entropy bonus (−c·log π(y) added to the reward; TRL's `entropy_coef`) buys
diversity back for a little accuracy. In this toy the collapse happens with or without clip-higher; DAPO
reports clip-higher's entropy effect on real models (primer §4).

## Worked example 5 — the length bias of averaging per sequence
A thinking policy that answers with probability 0.1 per step; completions that reach 16 thinking tokens are
truncated (no answer, reward 0). How are per-token losses turned into one number? TRL's `loss_type`:

In [ ]:
lengths = [10, 50]
for lt in ("grpo", "dapo", "dr_grpo"):
    wts = grpo.token_weights(lengths, lt, max_len=100)
    print(f"{lt:8}: weight per token of a 10-token completion {wts[0][0]:.4f}, of a 50-token one {wts[1][0]:.4f}")

Under `"grpo"` (mean per sequence, then over sequences) each token of the short answer counts five times as
much. For a correct answer that favours short ones; for a wrong answer it means a long wrong answer is punished
*less per token* — including a truncated one. Average the update over many groups at a fixed policy and compare
its direction with the exact gradient of accuracy:

In [ ]:
think = ThinkTask(e0=0.8, q=0.15, max_think=16)
tp = Policy.for_task(think)
tp.theta[:, 1] = math.log(0.1 / 0.9)
base = think.expected(tp.stop_probs(think))
exact = np.zeros_like(tp.theta)
for i in range(16):
    for j in range(2):
        up, dn = tp.copy(), tp.copy()
        up.theta[i, j] += 1e-5
        dn.theta[i, j] -= 1e-5
        exact[i, j] = (think.expected(up.stop_probs(think))["accuracy"] - think.expected(dn.stop_probs(think))["accuracy"]) / 2e-5
print(f"policy: accuracy {base['accuracy']:.3f}, mean thinking {base['length']:.1f}, truncated {base['truncated']:.3f}")
for lt in ("grpo", "dapo", "dr_grpo"):
    g = grpo.expected_update(tp, think, np.random.default_rng(0), lt, n_groups=600)
    stepped = tp.copy()
    stepped.step(g, 1e-3 / np.linalg.norm(g))
    after = think.expected(stepped.stop_probs(think))
    cos = (g * exact).sum() / np.linalg.norm(g) / np.linalg.norm(exact)
    print(f"{lt:8}: cos(update, true gradient) {cos:.2f}   along it, truncation "
          f"{'rises' if after['truncated'] > base['truncated'] else 'falls'} and mean length +{(after['length'] - base['length']) * 1e3:.2f} per 1e-3 step")

Token-level (`"dapo"`, TRL's default) and constant (`"dr_grpo"`) normalisers point along the true gradient and
cut truncation. The per-sequence mean points elsewhere: it lengthens thinking two to four times as fast and pushes
*into* truncation, because the truncated completions' penalty is spread over 16 tokens. That is the length bias,
and why DAPO also masks truncated completions (`mask_truncated_completions`) and shapes overlong ones.

## Worked example 6 — dynamic sampling: paying for groups that carry signal
A dataset of nine prompts: three the model nearly always solves, three it sometimes does, three it rarely
does. Each step trains on four prompts drawn at random.

In [ ]:
mixed = ThinkTask(prompts=[(0.02, 0.1)] * 3 + [(0.8, 0.15)] * 3 + [(0.995, 0.01)] * 3, max_think=16)
for ds in (False, True):
    h = grpo.train_grpo(Policy.for_task(mixed), mixed, np.random.default_rng(0),
                        grpo.GRPOConfig(num_generations=8, dynamic_sampling=ds, lr=20.0), steps=60,
                        batch_prompts=4, log_every=1)
    silent = np.mean([x["frac_reward_zero_std"] for x in h])
    made = np.mean([x["groups_generated"] for x in h])
    trained_silent = 0.0 if ds else silent
    print(f"dynamic_sampling={ds!s:5}: groups generated per step {made:4.1f}, of which silent {silent:.0%}; "
          f"silent groups in the trained batch {trained_silent:.0%}")

Without it, more than half of every batch is generated and then contributes nothing, so the effective batch
size wanders from step to step. DAPO over-samples and keeps only informative groups — every trained batch is
full — and pays for it in rollouts (here about twice as many groups generated per step; TRL has no flag for
it and logs `frac_reward_zero_std`). In DAPO's ablation
this was the largest single gain (AIME 2024 avg@32: 42 → 50; naive GRPO 30; primer §4, verify).

## Worked example 7 — where the compute goes
One synchronous RL step: 512 prompts × 16 samples = 8,192 completions of a 7.6B model on 64 H100s, lognormal
lengths (median 4,000 tokens), capped at 20,480 — DAPO's batch shape. Generation is decode-bound and waits for
the longest completion; training is a compute-bound 6·N FLOPs per token plus two scoring passes.

In [ ]:
m7 = workload.Model("7.6B policy", 7.6, 28, 4, 128)
out_len = np.minimum(4000 * np.exp(0.7 * np.random.default_rng(0).standard_normal(8192)), 20480)
r = workload.rl_step_time(m7, workload.GPUS["H100"], 64, 500, out_len)
print(f"mean completion {out_len.mean():,.0f} tokens, longest {out_len.max():,.0f}")
print(f"generate {r['generate_s']:.0f} s, train {r['train_s']:.0f} s → rollouts are {r['rollout_fraction']:.0%} of the step; "
      f"the generation batch is {r['batch_occupancy']:.0%} full on average (the long tail decodes almost alone)")

The model is optimistic (perfect overlap within each phase, 50% MFU on decode FLOPs, 40% on training) and still
puts half the step into generation, most of it spent waiting on a few long completions. verl reports ~70% for
rollouts in DAPO-32B training (primer §8, verify). That is why RL frameworks embed vLLM or SGLang, and why they
move to one-step-off-policy and fully asynchronous rollouts — at the price of stale samples that the clipped
ratio (and importance-sampling corrections) must absorb.

## Exercise 3.1 — group advantages, as TRL computes them
`rewards` has shape (n_groups, G). Subtract each group's mean and divide by its std **with Bessel's correction**
plus 1e-4.

In [ ]:
def my_group_advantages(rewards):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
batch = [[1, 0, 0, 1], [1, 0, 0, 0], [0.5, 0.5, 0.5, 0.5]]
assert np.allclose(my_group_advantages(batch), grpo.group_advantages(batch))
assert np.allclose(my_group_advantages(batch)[0], [0.865875, -0.865875, -0.865875, 0.865875], atol=1e-6)
print("✅ ±0.866 for a 2-of-4 group; zeros for a group that all scored the same")

## Exercise 3.2 — k3
Implement k3 from per-token log-probabilities, and confirm it is never negative on `d_grid`.

In [ ]:
def my_k3(logp, ref_logp):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
d_grid = np.linspace(-5, 5, 101)
assert np.allclose(my_k3(0.0, d_grid), grpo.k3(0.0, d_grid)) and my_k3(0.0, d_grid).min() >= 0
assert round(float(my_k3(0.0, 0.1)), 7) == 0.0051709
print("✅ k3 = e^d − d − 1 ≥ 0 for every d (e^d ≥ 1 + d), zero only when π = π_ref")

## Exercise 3.3 — which tokens get a gradient?
Return a boolean array: True where the clipped surrogate's gradient with respect to ρ is nonzero.

In [ ]:
def has_gradient(ratio, adv, eps_low=0.2, eps_high=0.2):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
rr = np.array([0.5, 0.79, 0.81, 1.0, 1.19, 1.21, 1.3, 1.5])
for a in (1.0, -1.0):
    for hi in (0.2, 0.28):
        _, d = grpo.clipped_surrogate(rr, np.full(8, a), 0.2, hi)
        assert np.array_equal(has_gradient(rr, np.full(8, a), 0.2, hi), d != 0)
print("✅ a token stops learning once its ratio has moved past the clip in the direction its advantage wants")

## Exercise 3.4 — the three normalisers
For completion lengths `lengths` (one group, G of them), return a list of per-token weight arrays for
`loss_type` in "grpo" (1/(G·|o_i|)), "dapo" (1/Σ|o|) and "dr_grpo" (1/(G·max_len)).

In [ ]:
def my_token_weights(lengths, loss_type, max_len):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
for lt in ("grpo", "dapo", "dr_grpo"):
    for a, b in zip(my_token_weights([3, 7, 20], lt, 32), grpo.token_weights([3, 7, 20], lt, 32)):
        assert np.allclose(a, b)
print("✅ per-sequence, token-level, constant: the same losses, three different ideas of what one token is worth")

## Exercise 3.5 — DAPO's soft overlong punishment
L_max = 100, cache 20: 0 up to 80 tokens, falling linearly to −1 at 100, −1 beyond.

In [ ]:
def my_overlong(n, max_len=100, cache=20):
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
assert [my_overlong(n) for n in (80, 90, 100, 101)] == [0.0, -0.5, -1.0, -1.0]
assert all(my_overlong(n) == grpo.soft_overlong_penalty(n, 100, 20) for n in range(0, 130))
print("✅ a graded penalty inside the last 20 tokens, so 'nearly too long' is a gradient, not a cliff")

## Exercise 3.6 — write the configs
Fill two `GRPOConfig`s. `dapo_cfg`: DAPO's recipe — clip-higher (0.2/0.28), token-level loss, no KL,
overlong filtering, soft overlong punishment with a cache of 4,096, dynamic sampling, G = 16.
`r1_cfg`: GRPO as the DeepSeek-R1 paper wrote it — per-sequence mean, group std scaling, ε = 0.2 both sides,
β = 0.001. (TRL spells DAPO's overlong filtering `mask_truncated_completions`.)

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert (dapo_cfg.epsilon, dapo_cfg.epsilon_high, dapo_cfg.loss_type, dapo_cfg.beta) == (0.2, 0.28, "dapo", 0.0)
assert dapo_cfg.mask_truncated_completions and dapo_cfg.dynamic_sampling and dapo_cfg.overlong_cache == 4096
assert dapo_cfg.num_generations == 16
assert (r1_cfg.beta, r1_cfg.loss_type, r1_cfg.scale_rewards, r1_cfg.epsilon_high) == (0.001, "grpo", "group", None)
print("✅ TRL's defaults are neither: beta=0.0, loss_type='dapo', no clip-higher, no masking — set them explicitly "
      "when you mean to reproduce a paper")

## In a design review
**The two-minute version.** "For tasks we can check — math answers, unit tests, output formats — we use RL with
verifiable rewards and GRPO. Each prompt gets a group of G samples; the group mean is the baseline, so there is
no value model and memory holds the policy, a reference only if β > 0, and the rollout engine. Groups that are
all right or all wrong carry no gradient, so we watch frac_reward_zero_std and resample or curate prompts to
stay near 50% solvable. We use the token-level loss — per-sequence averaging under-weights long completions and
drifts the model into long, truncated answers — mask truncated completions and add a soft overlong penalty
near the cap. The clip matters only with several updates per batch or off-policy rollouts. Most of each step
is generation, decode-bound and waiting on the longest completion, so the rollout side is an inference-serving
problem: KV capacity, batching, and async rollouts with bounded staleness."

**Drill questions**
1. *Why does GRPO not need a value model?* — The baseline is the mean reward of G samples of the same prompt;
   PPO learns V(s) to get per-token baselines, a second network as large as the policy.
2. *Loss falls to zero and accuracy plateaus at 95% on the training set. What is happening?* — Most groups are
   all-correct, so advantages are zero: no signal. Harder prompts, dynamic sampling, or a curriculum.
3. *Responses keep getting longer and more of them hit max_completion_length. Which setting would you check
   first?* — `loss_type`: per-sequence ("grpo") averaging penalises long wrong answers less per token; use
   "dapo"/"dr_grpo", mask truncated completions, add the soft overlong penalty.